In [2]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 201.6 kB/s  0:01:30m0:00:0100:08


In [8]:
import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist, squareform
import gurobipy as gp
from gurobipy import GRB

class FPLHRP:
    """
    Hierarchical Risk Parity (HRP) adapted for FPL Squad Selection.
    Uses expected points (xP) as a 'return' signal and historical
    variance/correlation to manage risk (consistency).
    """

    def __init__(self):
        self.weights = None

    def get_quasi_diag(self, link):
        """Reorder the covariance matrix (Stage 2)"""
        link = link.astype(int)
        sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
        num_items = link[-1, 3]

        while sort_ix.max() >= num_items:
            sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
            df0 = sort_ix[sort_ix >= num_items]
            i = df0.index
            j = df0.values - num_items
            sort_ix[i] = link[j, 0]
            df0 = pd.Series(link[j, 1], index=i + 1)
            sort_ix = pd.concat([sort_ix, df0])
            sort_ix = sort_ix.sort_index()
            sort_ix.index = range(sort_ix.shape[0])

        return sort_ix.tolist()

    def get_cluster_var(self, cov, cluster_items):
        """Compute variance of a cluster using inverse-variance weighting"""
        cov_slice = cov.loc[cluster_items, cluster_items]
        ivp = 1.0 / np.diag(cov_slice)
        ivp /= ivp.sum()
        w = ivp.reshape(-1, 1)
        cluster_var = np.dot(np.dot(w.T, cov_slice), w)[0, 0]
        return cluster_var

    def get_recursive_bisection(self, cov, sort_ix, xp_signals=None):
        """Recursive Bisection (Stage 3) - Augmented with xP signals"""
        w = pd.Series(1, index=sort_ix)
        c_items = [sort_ix]

        while len(c_items) > 0:
            c_items = [i[j:k] for i in c_items for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) if len(i) > 1]
            for i in range(0, len(c_items), 2):
                c_items0 = c_items[i]
                c_items1 = c_items[i+1]

                var0 = self.get_cluster_var(cov, c_items0)
                var1 = self.get_cluster_var(cov, c_items1)

                # Base Risk-Parity Alpha
                alpha = 1 - var0 / (var0 + var1)

                # OPTIONAL: Tilt weights towards higher xP if provided
                if xp_signals is not None:
                    xp0 = xp_signals.loc[c_items0].mean()
                    xp1 = xp_signals.loc[c_items1].mean()
                    tilt = (xp0 / (xp0 + xp1)) * 0.2
                    alpha = np.clip(alpha + tilt - 0.1, 0.05, 0.95)

                w[c_items0] *= alpha
                w[c_items1] *= (1 - alpha)
        return w

    def optimize(self, returns_df, xp_df):
        """
        returns_df: DataFrame of historical point returns (for correlation)
        xp_df: Series of xP values indexed by player ID
        """
        corr = returns_df.corr().fillna(0)
        cov = returns_df.cov().fillna(0)

        # Distance matrix (correlation-based)
        dist = np.sqrt(0.5 * (1 - corr))

        # 1. Tree Clustering
        link = sch.linkage(squareform(dist), method='single')

        # 2. Quasi-Diagonalization
        sort_ix = self.get_quasi_diag(link)
        sort_ix = corr.index[sort_ix].tolist()

        # 3. Recursive Bisection
        self.weights = self.get_recursive_bisection(cov, sort_ix, xp_signals=xp_df)
        return self.weights

def select_best_squad(players_df, budget=100.0, team_limit=3):
    """
    Selects the best 15-player squad (including bench) given HRP scores.
    Uses Gurobi (gurobipy) to solve the constrained knapsack problem.
    """
    try:
        # Create a new model
        model = gp.Model("FPL_Squad_Selection")
        model.Params.LogToConsole = 0 # Silent mode

        # Decision variables: 1 if player is selected, 0 otherwise
        player_vars = model.addVars(players_df.index, vtype=GRB.BINARY, name="player")

        # Objective: Maximize HRP-weighted xP score
        obj = gp.quicksum(players_df.loc[i, 'hrp_score'] * player_vars[i] for i in players_df.index)
        model.setObjective(obj, GRB.MAXIMIZE)

        # Constraints
        # 1. Total players = 15
        model.addConstr(gp.quicksum(player_vars[i] for i in players_df.index) == 15, "total_players")

        # 2. Position constraints (2 GKP, 5 DEF, 5 MID, 3 FWD)
        model.addConstr(gp.quicksum(player_vars[i] for i in players_df.index if players_df.loc[i, 'position'] == 'GKP') == 2, "gkp_count")
        model.addConstr(gp.quicksum(player_vars[i] for i in players_df.index if players_df.loc[i, 'position'] == 'DEF') == 5, "def_count")
        model.addConstr(gp.quicksum(player_vars[i] for i in players_df.index if players_df.loc[i, 'position'] == 'MID') == 5, "mid_count")
        model.addConstr(gp.quicksum(player_vars[i] for i in players_df.index if players_df.loc[i, 'position'] == 'FWD') == 3, "fwd_count")

        # 3. Budget constraint
        model.addConstr(gp.quicksum(players_df.loc[i, 'price'] * player_vars[i] for i in players_df.index) <= budget, "budget_limit")

        # 4. Team limit (max 3 players per team)
        teams = players_df['team'].unique()
        for t in teams:
            model.addConstr(gp.quicksum(player_vars[i] for i in players_df.index if players_df.loc[i, 'team'] == t) <= team_limit, f"team_limit_{t}")

        # Optimize model
        model.optimize()

        if model.Status == GRB.OPTIMAL:
            selected_ids = [i for i in players_df.index if player_vars[i].X > 0.5]
            return players_df.loc[selected_ids]
        else:
            return None

    except gp.GurobiError as e:
        print(f"Error reported by Gurobi: {e}")
        return None

def run_fpl_optimization(players_df, hist_returns, budget=100.0):
    """
    Wrapper for full HRP -> Constrained Selection workflow.
    players_df columns: [fpl_id, fpl_name, xP_preds, team, position, price]
    """
    # 1. Align data
    valid_ids = list(set(players_df['fpl_id']) & set(hist_returns.columns))
    players_data = players_df[players_df['fpl_id'].isin(valid_ids)].copy().set_index('fpl_id')
    hist_returns = hist_returns[valid_ids]

    # 2. Get HRP Weights
    hrp = FPLHRP()
    weights = hrp.optimize(hist_returns, players_data['xP_preds'])

    # 3. Combine HRP Weight with xP for a selection score
    players_data['hrp_score'] = weights * players_data['xP_preds']

    # 4. Solve for optimal 15-man squad
    squad = select_best_squad(players_data, budget=budget)

    return squad

# Mock Data Generation
# Create a larger pool of mock players
import random
positions = ['GKP']*10 + ['DEF']*30 + ['MID']*30 + ['FWD']*20
teams = ['ARS', 'MCI', 'LIV', 'CHE', 'AVL', 'MUN', 'TOT', 'NEW']

mock_data = {
    'fpl_id': list(range(1, 91)),
    'fpl_name': [f'Player_{i}' for i in range(1, 91)],
    'xP_preds': [random.uniform(2.0, 9.0) for _ in range(90)],
    'team': [random.choice(teams) for _ in range(90)],
    'position': positions,
    'price': [random.uniform(4.0, 12.5) for _ in range(90)]
}
mock_players_df = pd.DataFrame(mock_data)

# Mock Returns
np.random.seed(42)
mock_returns = pd.DataFrame(
    np.random.randn(10, 90),
    columns=list(range(1, 91))
)

final_squad = run_fpl_optimization(mock_players_df, mock_returns)

if final_squad is not None:
    print(f"Optimal 15-man Squad (Budget Used: {final_squad['price'].sum():.1f}m):")
    print(final_squad[['fpl_name', 'position', 'team', 'price', 'xP_preds']].sort_values('position'))
else:
    print("Could not find an optimal squad within constraints.")

/tmp/ipykernel_526959/3856710563.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312 0.74862312
 0.74862312 0.74862312 0.74862312]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  w[c_items0] *= alpha


Set parameter LogToConsole to value 0
Optimal 15-man Squad (Budget Used: 99.8m):
         fpl_name position team     price  xP_preds
fpl_id                                             
11      Player_11      DEF  MUN  6.068805  6.182432
16      Player_16      DEF  NEW  5.806627  8.244554
18      Player_18      DEF  NEW  5.602963  4.023455
20      Player_20      DEF  MCI  8.803476  8.031448
30      Player_30      DEF  AVL  7.172933  8.653866
72      Player_72      FWD  NEW  7.930613  8.733632
74      Player_74      FWD  AVL  7.416243  4.937913
88      Player_88      FWD  MCI  5.018433  6.492355
4        Player_4      GKP  CHE  8.868734  6.571846
9        Player_9      GKP  CHE  4.660606  8.547922
43      Player_43      MID  MCI  5.087602  6.932803
47      Player_47      MID  MUN  6.930684  2.472624
59      Player_59      MID  TOT  5.293775  6.148277
65      Player_65      MID  CHE  9.325294  7.863594
68      Player_68      MID  TOT  5.859069  8.928726


In [11]:
final_squad

,fpl_name,xP_preds,team,position,price,hrp_score
fpl_id,,,,,,
4,Player_4,6.571846,CHE,GKP,8.868734,0.134597
9,Player_9,8.547922,CHE,GKP,4.660606,0.086135
11,Player_11,6.182432,MUN,DEF,6.068805,0.078826
16,Player_16,8.244554,NEW,DEF,5.806627,0.207582
18,Player_18,4.023455,NEW,DEF,5.602963,0.299821
20,Player_20,8.031448,MCI,DEF,8.803476,0.195568
30,Player_30,8.653866,AVL,DEF,7.172933,0.114749
43,Player_43,6.932803,MCI,MID,5.087602,0.046406
47,Player_47,2.472624,MUN,MID,6.930684,0.180273


In [ ]:
import requests
import pandas as pd
import numpy as np
import io

# API Configuration
BASE_URL = "https://fantasy.premierleague.com/api/"

# External Data Sources (GitHub)
VAASTAV_URL = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/"
CORE_INSIGHTS_URL = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/master/data/"

def fetch_data(endpoint):
    """Generic helper to fetch data from FPL API"""
    try:
        response = requests.get(f"{BASE_URL}{endpoint}", timeout=10)
        return response.json()
    except Exception as e:
        print(f"Error fetching data from {endpoint}: {e}")
        return {}

def get_player_data():
    """Fetches all players and their current 25/26 season totals/metrics"""
    data = fetch_data("bootstrap-static/")
    if not data:
        return pd.DataFrame()

    elements = pd.DataFrame(data['elements'])
    element_types = pd.DataFrame(data['element_types'])

    # Map position names
    elements['position'] = elements['element_type'].map(
        element_types.set_index('id')['singular_name_short']
    )

    # Clean up numeric columns for current season
    cols_to_fix = ['expected_goals', 'expected_assists', 'expected_goal_involvements',
                   'expected_conceded', 'form', 'points_per_game', 'ict_index']
    for col in cols_to_fix:
        if col in elements.columns:
            elements[col] = pd.to_numeric(elements[col], errors='coerce').fillna(0)

    return elements

class HistoricalDataSource:
    """
    Handles fetching and cleaning data from external repositories
    (Vaastav and olbauday/FPL-Core-Insights).
    """
    @staticmethod
    def get_24_25_data():
        """Fetches the full 24/25 season match-by-match data"""
        url = f"{VAASTAV_URL}2024-25/gws/merged_gw.csv"
        try:
            print(f"Fetching 24/25 historical data from Vaastav repo...")
            response = requests.get(url, timeout=15)
            df = pd.read_csv(io.StringIO(response.text))
            return df
        except Exception as e:
            print(f"Error loading historical CSV: {e}")
            return pd.DataFrame()

class XPEngine:
    """
    Calculates Expected Points (xP) for the 2025/26 season rules,
    integrating granular data from external historical repositories.
    """
    POINTS_MAP = {
        'GKP': {'goal': 10, 'assist': 3, 'clean_sheet': 4},
        'DEF': {'goal': 6, 'assist': 3, 'clean_sheet': 4},
        'MID': {'goal': 5, 'assist': 3, 'clean_sheet': 1},
        'FWD': {'goal': 4, 'assist': 3, 'clean_sheet': 0}
    }

    def __init__(self, current_gw=20):
        self.current_gw = current_gw
        self.past_data_24_25 = HistoricalDataSource.get_24_25_data()

    def calculate_xp(self, player_row, cs_prob=0.3):
        """
        Calculates xP by merging granular 25/26 history and 24/25 match-level data.
        """
        pos = player_row['position']
        player_id = player_row['id']
        player_name = player_row['second_name']

        # Fetch individual history from official API for 25/26
        summary = fetch_data(f"element-summary/{player_id}/")

        # --- 1. CURRENT SEASON DATA (25/26: GW1-20) ---
        history_25_26 = pd.DataFrame(summary.get('history', []))
        if not history_25_26.empty:
            curr_minutes = history_25_26['minutes'].sum()
            factor = 90 / max(curr_minutes, 1)
            curr_xg_90 = history_25_26['expected_goals'].apply(pd.to_numeric).sum() * factor
            curr_xa_90 = history_25_26['expected_assists'].apply(pd.to_numeric).sum() * factor
            curr_ict_90 = history_25_26['ict_index'].apply(pd.to_numeric).sum() * factor
        else:
            curr_xg_90 = (player_row['expected_goals'] / max(player_row['minutes'], 1)) * 90
            curr_xa_90 = (player_row['expected_assists'] / max(player_row['minutes'], 1)) * 90
            curr_ict_90 = (player_row['ict_index'] / max(player_row['minutes'], 1)) * 90

        # --- 2. GRANULAR HISTORICAL DATA (24/25) ---
        # We try to find the player in Vaastav's 24/25 dataset
        # In Vaastav data, 'element' usually maps to FPL ID
        past_xg_90, past_xa_90 = curr_xg_90, curr_xa_90 # Default fallback

        if not self.past_data_24_25.empty:
            # Note: We use 'element' column which is the unique player ID in FPL API
            player_past = self.past_data_24_25[self.past_data_24_25['element'] == player_id]

            if not player_past.empty:
                past_min = player_past['minutes'].sum()
                if past_min > 90:
                    past_factor = 90 / past_min
                    past_xg_90 = player_past['expected_goals'].sum() * past_factor
                    past_xa_90 = player_past['expected_assists'].sum() * past_factor

        # --- 3. WEIGHTED INTEGRATION ---
        # Using a 65% weight for the 20 matches of 25/26 and 35% for the 38 matches of 24/25
        avg_xg_90 = (curr_xg_90 * 0.65) + (past_xg_90 * 0.35)
        avg_xa_90 = (curr_xa_90 * 0.65) + (past_xa_90 * 0.35)

        # --- 4. GW21 xP PROJECTION ---
        proj_minutes_factor = 85 / 90 # Assuming a starter plays 85 mins

        # Appearance
        xP_appearance = 2.0 if float(player_row['form']) > 1 else 0.5

        # Attacking Points (Position-specific)
        xP_attack = ((avg_xg_90 * self.POINTS_MAP[pos]['goal']) +
                     (avg_xa_90 * self.POINTS_MAP[pos]['assist'])) * proj_minutes_factor

        # Defensive Points (Clean Sheet)
        xP_defensive = cs_prob * self.POINTS_MAP[pos]['clean_sheet']

        # 2025/26 CBIT Rule Adjustment (Defensive Contributions)
        # Using volume metrics to estimate the chance of +2 points
        cbit_prob = 0.8 if curr_ict_90 > 8.5 else (0.2 if curr_ict_90 < 4 else 0.45)
        xP_cbit = cbit_prob * 2.0

        # Bonus projection (Refined for 25/26 BPS changes)
        xP_bonus = (avg_xg_90 + avg_xa_90) * 1.4 + (cbit_prob * 0.6)

        total_xp = xP_appearance + xP_attack + xP_defensive + xP_cbit + xP_bonus
        return round(total_xp, 2)

# if __name__ == "__main__":
print("FPL xP Predictor v2.1: Rule-Aware Analysis")
print("Data Sources: Official API, Vaastav (24/25), olbauday (25/26 Stats)")

players = get_player_data()
if players.empty:
    print("Failed to load player data.")
else:
    # Initialize Engine (fetches historical CSV on init)
    engine = XPEngine(current_gw=20)

    # Filter for active and available players
    active_players = players[(players['minutes'] > 300) & (players['status'] == 'a')].copy()

    print(f"Calculating xP for {len(active_players)} active players for GW21...")

    # Limiting to top 60 to demonstrate without extreme API delay
    top_candidates = active_players.sort_values('form', ascending=False).head(60).copy()

    top_candidates['xP_GW21'] = top_candidates.apply(
        lambda row: engine.calculate_xp(row, cs_prob=0.3), axis=1
    )

    print("\n--- GW21 HYBRID xP PROJECTIONS (Top 15) ---")
    results = top_candidates[['second_name', 'position', 'xP_GW21', 'now_cost', 'form']]
    print(results.sort_values('xP_GW21', ascending=False).head(15).to_string(index=False))

    print("\n--- HIGH-CEILING DIFFERENTIALS (Ownership < 10%) ---")
    # Assuming we have selected_by_percent from bootstrap-static
    top_candidates['selected_by_percent'] = pd.to_numeric(top_candidates['selected_by_percent'])
    differentials = top_candidates[top_candidates['selected_by_percent'] < 10]
    print(differentials.sort_values('xP_GW21', ascending=False).head(5)[['second_name', 'xP_GW21', 'selected_by_percent']].to_string(index=False))

FPL xP Predictor v2.1: Rule-Aware Analysis
Data Sources: Official API, Vaastav (24/25), olbauday (25/26 Stats)
Fetching 24/25 historical data from Vaastav repo...
Calculating xP for 299 active players for GW21...

--- GW21 HYBRID xP PROJECTIONS (Top 15) ---
             second_name position  xP_GW21  now_cost  form
                 Haaland      FWD     9.07       151   6.8
                  Cherki      MID     7.17        68   5.5
                    Saka      MID     6.99       102   5.7
               Fernández      MID     6.86        64   4.3
           Calvert-Lewin      FWD     6.25        60   6.8
    Nascimento Rodrigues      FWD     6.24        71   6.7
                Robinson      DEF     6.19        49   4.3
                   Wirtz      MID     6.18        82   6.5
                 Semenyo      MID     6.08        76   6.2
                Ødegaard      MID     5.80        78   4.8
                 Collins      DEF     5.74        50   8.0
Santos Carneiro da Cunha      MID 

In [20]:
players

,can_transact,can_select,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,dreamteam_count,...,form_rank,form_rank_type,points_per_game_rank,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,clean_sheets_per_90,defensive_contribution_per_90,position
0,True,True,NaN,NaN,154561,-1,1,4,-4,1,...,105,9,71,4,6,1,1.00,0.48,0.00,GKP
1,True,True,NaN,NaN,109745,0,0,-4,4,0,...,541,76,607,81,270,36,0.00,0.00,0.00,GKP
2,True,False,0.0,0.0,463748,0,0,0,0,0,...,499,66,568,71,359,52,0.00,0.00,0.00,GKP
3,True,True,NaN,NaN,551221,0,0,-1,1,0,...,469,50,537,55,337,47,0.00,0.00,0.00,GKP
4,True,True,100.0,100.0,226597,1,-1,7,-7,4,...,26,6,2,1,9,2,1.00,0.64,8.75,DEF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
790,True,True,NaN,NaN,647671,0,0,0,0,1,...,27,5,129,11,219,33,0.81,0.20,6.66,FWD
791,True,True,NaN,NaN,601956,0,0,0,0,0,...,529,211,597,231,721,246,0.00,0.00,0.00,DEF
792,True,True,NaN,NaN,605319,0,0,0,0,0,...,673,273,715,298,752,327,0.00,0.00,0.00,MID
793,True,True,NaN,NaN,514362,0,0,0,0,0,...,642,249,685,275,779,344,0.00,0.00,0.00,MID
